# Présenttation des outils

Deux outils pour réaliser le Web Scrapping:

- **BeautifulSoup** : 
  - Bibliothèque Python pour extraire des données de documents HTML et XML.
  - Permet une navigation facile dans les structures des pages.
    
- **Requests** : 
  - Bibliothèque Python pour effectuer des requêtes HTTP.
  - Utilisée pour récupérer le contenu de la page web.
 - **lxml** : 
  - pour mieux gérer les codes html
  - plus rapide
  

# Importation des packages

In [1]:
!pip install beautifulsoup4
!pip install requests
!pip install lxml
!pip install html5lib

  Using cached lxml-6.1.1-cp312-cp312-win_amd64.whl.metadata (3.6 kB)
Using cached lxml-6.1.1-cp312-cp312-win_amd64.whl (4.0 MB)


In [1]:
from bs4 import BeautifulSoup
import requests
import lxml
import pandas as pd
import time
import random as ran
import sys

# Objectif du projet


Dans ce projet, nous allons préparer la base de donnée devant constitué l'annuaire des entreprises faisant du call center comme activité principale de soutien aux entreprise.  Nous allons récupérer les informations suivantes à partir du site :

1. **numéro de téléphone**
2. **Adresse de l'entreprise**
3. **Nom de l'entreprise** 


# Web scraping

In [2]:
url='https://www.goafricaonline.com/sn/annuaire/centres-appels-call-center'

**On récupère le contenu brut de la page, c'est à dire le code html**

In [3]:
source = requests.get(url)

In [4]:
source

<Response [200]>

**On utilise l'analyseur de syntaxe 'lxml' que BeautifulSoup va utiliser pour interpréter le HTML et construire l'arbre de navigation**

In [5]:
soup = BeautifulSoup(source.content,'lxml')


In [6]:
blocs_entreprises = soup.find_all('article', {'data-role': 'company'})
print(f"{len(blocs_entreprises)} entreprises trouvées sur cette page")


30 entreprises trouvées sur cette page


In [7]:
nom_entreprise = blocs_entreprises[0].find('a').getText()
nom_entreprise


'ABACIS GROUPE SARL'

In [ ]:
# Récupérer le numéro de téléphone pour joindre l'entreprise
# On cible le div contenant l'icône téléphone (tnp-smartphone-2)
div_gsm = blocs_entreprises[2].find('i', class_='tnp-smartphone-2').find_parent('div')
# On récupère le lien tel: à l'intérieur de ce div
lien_tel = div_gsm.find('a', href=lambda h: h and h.startswith('tel:'))
if lien_tel:
    numero_complet = lien_tel['href'].replace('tel:', '').strip()
    numero_complet_propre = numero_complet.replace(' ', '')
else:
    numero_complet_propre = None

#récupérer l'adresse de l'entreprise
adresse = blocs_entreprises[0].find('address', class_='text-14 text-gray-700 flex-1').text.strip()

# Récupérer le nom de l'entreprise
nom_entreprise = blocs_entreprises[0].find('a').getText()


# Afficher toutes les infos de l'entreprise
print(
      "Nom entreprise : " + nom_entreprise,
      "\nNuméro de téléphone : " + numero_complet_propre,
      "\nAdresse : " + adresse)
      


Nom entreprise : ABACIS GROUPE SARL 
Numéro de téléphone : +22177710995 
Adresse : MermozDakar - 10700 - Sénégal 
Description : ABACIS GROUPE c'est une société de prestation de service informatique et centre d'appel. Notre centre d'appel accueil vos clients dans la résolution de leur souci et le traitement de leur demande. Nous nous occupons de vos prospects, afin de générer plus de leads vers vos canaux, quelque soit leur nature


In [41]:
def recuperer_info_entreprise(blocs_entreprise):
    """
    Récupère les informations relatives à une entreprise située sur un bloc de données HTML fourni.

    Paramètres :
        blocs_entreprise : Le bloc HTML contenant les info de l'entreprise de call center.

    Retourne :
        dict : Un dictionnaire contenant les informations sur l'entreprise. Les clés du dictionnaire sont :
            - 'nom' : la raison sociale de l'entreprise.
            - 'numero' : Le numéro de téléphone permettant de joindre l'entreprise.
            - 'adresse' : L'adresse physique de l'entreprise.
            
    """

    # Initialiser un dictionnaire pour stocker les données de l'entreprise
    infos_entreprise = {}

    # Récupérer la raison sociale
    try:
      infos_entreprise['nom'] = blocs_entreprise.find('a').getText()
    except:
      infos_entreprise['nom'] = None

    # Récupérer le numero
    # On cible le div contenant l'icône téléphone (tnp-smartphone-2)
    div_gsm = blocs_entreprises[2].find('i', class_='tnp-smartphone-2').find_parent('div')
    # On récupère le lien tel: à l'intérieur de ce div car il représente le numéro
    lien_tel = div_gsm.find('a', href=lambda h: h and h.startswith('tel:'))
    if lien_tel:
      numero_complet = lien_tel['href'].replace('tel:', '').strip() # on enlève le tel qu'il y a au début du lien href
      numero_complet_propre = numero_complet.replace(' ', '')
    else:
      numero_complet_propre = None
    infos_entreprise['numero']=numero_complet_propre
    # Récupérer l'adresse
    try:
      infos_entreprise['adresse'] = blocs_entreprise.find('address', class_='text-14 text-gray-700 flex-1').text.strip()

    except:
      infos_entreprise['adresse'] = None



    # Retourner le dictionnaire contenant les info de l'entreprise
    return infos_entreprise


In [42]:
def extraire_donnees_page_entreprise(blocs_d_entreprises):
    """
    Extrait les informations de chaque entreprise à partir des blocs de données HTML de la page.

    Paramètres :
        blocs_d_entreprises (list) : Une liste des blocs HTML contenant les détails de chaque entreprise.

    Retourne :
        list : Une liste de dictionnaires, chaque dictionnaire contenant les détails d'une entreprise.  Les clés du dictionnaire sont :
            - 'nom' : la raison sociale de l'entreprise.
            - 'numero' : Le numéro de téléphone permettant de joindre l'entreprise.
            - 'adresse' : L'adresse physique de l'entreprise.
           
    """

    liste_donnees_entreprises = []  # Initialiser une liste pour stocker les détails de chaque entreprise 
    nb_blocs = len(blocs_d_entreprises)

    for i in range(nb_blocs):
        # Appeler la fonction récupérant les infos de l'entreprise pour chaque bloc d'entreprise
        donnees_entreprise = recuperer_info_entreprise(blocs_d_entreprises[i])

        # Ajouter les infos de l'entreprise à la liste des données de la page
        liste_donnees_entreprises.append(donnees_entreprise)

    return liste_donnees_entreprises


In [43]:

def extraire_donnees_entreprises_par_lien(url):
    """
    Extrait les infos des entreprises à partir d'un lien donné pour un nombre de page i donné.

    Paramètres :
        url (str) : L'URL de la page à partir de laquelle extraire les infos des entreprises.
    Retourne :
        list : Une liste de dictionnaires, chaque dictionnaire contenant les infos d'une entreprise. Les clés du dictionnaire sont :
            - 'nom' : la raison sociale de l'entreprise.
            - 'numero' : Le numéro de téléphone permettant de joindre l'entreprise.
            - 'adresse' : L'adresse physique de l'entreprise.
            - 'description' : Description de l'activité de l'entreprise.
            
    """

    url_base = url
    donnees_entreprises = []  # Initialiser une liste pour stocker les infos de chaque entreprise
    
    for i in range(1,3): # il y a 2 pages
        url_complete = url_base + str(i)
        
        # Récupérer le contenu HTML de la page
        source = requests.get(url_complete).text
        soup = BeautifulSoup(source, 'html.parser')
        
        # Trouver tous les blocs d'entreprises sur la page
        blocs_de_entreprises = soup.find_all('article', {'data-role': 'company'})
        
        # Appeler la fonction extraire_donnees_page_entreprise pour chaque page d'entreprises
        donnees_page_entreprises = extraire_donnees_page_entreprise(blocs_de_entreprises)

        # Ajouter les infos des entreprises à la liste de données des entreprises
        donnees_entreprises.extend(donnees_page_entreprises)   
        
        # Attendre un certain temps aléatoire avant de récupérer la prochaine page (pour éviter de surcharger le serveur)
        time.sleep(ran.randint(0, 10))
    
    return donnees_entreprises


In [44]:
base_scraping_link = "https://www.goafricaonline.com/sn/annuaire/centres-appels-call-center?p="

# Appeler la fonction pour extraire les infos des entreprises à partir du lien de base et avec le nombre de entreprises cible
entreprises = extraire_donnees_entreprises_par_lien(base_scraping_link)

# Créer un DataFrame à partir de la liste des infos des entreprises
base = pd.DataFrame(entreprises)
base.head(5)

,nom,numero,adresse
0,ABACIS GROUPE SARL,+221770542490,MermozDakar - 10700 - Sénégal
1,AD CENTER SERVICES,+221770542490,Route des AlmadiesDakar - Sénégal
2,AD CENTER SERVICES Almadies,+221770542490,Rte des Almadies - NgorDakarDakar - Sénégal
3,AD CENTER SERVICES FANN,+221770542490,- - Fann-Point E-AmitiéDakarDakar - Sénégal
4,AFRICATEL AVS,+221770542490,Mermoz Extension VDN lot numéro 19Dakar - Sénégal


In [45]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   nom      59 non-null     object
 1   numero   59 non-null     object
 2   adresse  59 non-null     object
dtypes: object(3)
memory usage: 1.5+ KB


In [46]:
base.to_csv('base.csv')